In [ ]:
# Relative path variables define local inputs and outputs.
NOTEBOOK_ID <- "Figure_4"

PAPER_EXACT_ROOT <- Sys.getenv(
    "PAPER_EXACT_ROOT",
    unset = "."
)
HISTORICAL_DATA_ROOT <- Sys.getenv(
    "HISTORICAL_DATA_ROOT",
    unset = "data/khm_scRNA"
)
RUN_ID <- Sys.getenv(
    "PAPER_EXACT_RUN_ID",
    unset = format(Sys.time(), "%Y%m%d_%H%M%S")
)
SAFE_OUTPUT_ROOT <- file.path(
    PAPER_EXACT_ROOT,
    "run_outputs_self_contained_20260826",
    paste0(NOTEBOOK_ID, "_", RUN_ID)
)
dir.create(SAFE_OUTPUT_ROOT, recursive = TRUE, showWarnings = FALSE)

options(
    stringsAsFactors = FALSE,
    repr.plot.res = 120,
    future.globals.maxSize = 20 * 1024^3
)

set.seed(1234)

# Figure 4 annotation.
RECOMPUTE_LONG_ANALYSES <- FALSE

suppressPackageStartupMessages({
    library(GeneNMF)
    library(Seurat)
    library(ggplot2)
    library(UCell)
    library(patchwork)
    library(Matrix)
    library(RcppML)
    library(viridis)
    library(dplyr)
    library(stringr)
    library(circlize)
    library(tidyr)
    library(tibble)
    library(ComplexHeatmap)
})

HUH7_SEURAT_RDS <- file.path(
    HISTORICAL_DATA_ROOT,
    "huh7_zxs",
    "result_zxs",
    "scdata_filter.rds"
)
stopifnot(file.exists(HUH7_SEURAT_RDS))

seu <- readRDS(HUH7_SEURAT_RDS)
seu <- subset(
    seu,
    subset = batch %in% c("normal2", "Tatin")
)
DefaultAssay(seu) <- "RNA"
seu <- NormalizeData(
    seu,
    normalization.method = "LogNormalize",
    verbose = FALSE
)
seurat_obj <- seu

cat("Raw Huh7 cells:", ncol(seu), "\n")
cat("Notebook output directory:", SAFE_OUTPUT_ROOT, "\n")


In [ ]:
seu.list <- SplitObject(
    seu,
    split.by = "batch"
)

geneNMF.programs <- multiNMF(
    seu.list,
    assay = "RNA",
    slot = "data",
    k = 5:10,
    nfeatures = 10000
)


In [ ]:
geneNMF.metaprograms <- getMetaPrograms(
    geneNMF.programs,
    nMP = 16,
    min.confidence = 0.8,
    weight.explained = 0.7,
    max.genes = 100,
    remove.empty = TRUE
)

geneNMF.metaprograms$metaprograms.metrics

mp.genes <- geneNMF.metaprograms$metaprograms.genes
seu <- AddModuleScore_UCell(
    seu,
    features = mp.genes,
    assay = "RNA",
    ncores = 2,
    name = ""
)

mp_matrix <- as.matrix(
    seu@meta.data[, names(mp.genes), drop = FALSE]
)
colnames(mp_matrix) <- paste0("MP_", seq_len(ncol(mp_matrix)))
seu@reductions[["MPsignatures"]] <- new(
    "DimReduc",
    cell.embeddings = mp_matrix,
    assay.used = "RNA",
    key = "MP_",
    global = FALSE
)

set.seed(123)
seu <- RunUMAP(
    seu,
    reduction = "MPsignatures",
    dims = seq_len(ncol(mp_matrix)),
    metric = "euclidean",
    reduction.name = "umap_MP",
    seed.use = 123,
    verbose = FALSE
)
seu <- FindNeighbors(
    seu,
    reduction = "MPsignatures",
    dims = seq_len(ncol(mp_matrix)),
    verbose = FALSE
)
seu <- FindClusters(
    seu,
    resolution = 0.7,
    verbose = FALSE
)


## Figure 4a


In [ ]:
meta_data <- seu@meta.data
meta_data <- meta_data %>%
    mutate(annotation = case_when(
        RNA_snn_res.0.7 %in% c(4,6) ~ 'Subset 1',
        RNA_snn_res.0.7 %in% c(1,7) ~ 'Subset 2',
        RNA_snn_res.0.7 %in% c(3,5) ~ 'Subset 3',
        RNA_snn_res.0.7 %in% c(0,2) ~ 'Subset 4'
    ))
meta_data$annotation %>% table()
meta_data %>% head()
seu@meta.data <- meta_data

In [ ]:
options(repr.plot.width = 26,repr.plot.height = 10)
p1 <- DimPlot(seu, reduction = "umap_MP",group.by = c('annotation'),cols = c("#E59C83","#88A3D3","#7BA191","#9A8AB2"),label = FALSE,pt.size = 2.5) +

  theme(aspect.ratio = 1,
        axis.text = element_blank(),
        axis.title = element_blank(),
        axis.ticks = element_blank())
library(viridis)
p2 <- FeaturePlot(seu, features = c('MP12','MP16','MP1','MP7'), reduction = "umap_MP", ncol=2) &
  scale_color_viridis(option="B") &
  theme(aspect.ratio = 1, axis.text=element_blank(), axis.ticks=element_blank())
p1 | p2

## Figure 4b


In [ ]:
invisible(geneNMF.metaprograms)


### Figure 4


In [ ]:
meta_data <- seu@meta.data
meta_data <- meta_data %>%
    mutate(annotation = case_when(
        RNA_snn_res.0.7 %in% c(4,6) ~ 'Subset 1',
        RNA_snn_res.0.7 %in% c(1,7) ~ 'Subset 2',
        RNA_snn_res.0.7 %in% c(3,5) ~ 'Subset 3',
        RNA_snn_res.0.7 %in% c(0,2) ~ 'Subset 4'
    ))
meta_data$annotation %>% table()
meta_data %>% head()
seu@meta.data <- meta_data

In [ ]:
suppressPackageStartupMessages({
    library(msigdbr)
    library(fgsea)
})

top_p_GOBP <- lapply(
    geneNMF.metaprograms$metaprograms.genes,
    function(program) {
        runGSEA(
            program,
            universe = rownames(seu),
            category = "C5",
            subcategory = "BP"
        )
    }
)
names(top_p_GOBP) <- names(geneNMF.metaprograms$metaprograms.genes)


In [ ]:
go_list <- list(
  MP12 = c(
      "GOBP_STEROL_BIOSYNTHETIC_PROCESS","GOBP_STEROL_METABOLIC_PROCESS","GOBP_LIPID_METABOLIC_PROCESS","GOBP_CELLULAR_RESPONSE_TO_INSULIN_STIMULUS"
  ),
  MP16 = c(
      "GOBP_REGULATION_OF_TRANSFERASE_ACTIVITY","GOBP_LOCOMOTION","GOBP_CELL_MIGRATION","GOBP_REGULATION_OF_PROTEIN_PHOSPHORYLATION"
  ),
  MP1 = c(
      "GOBP_MITOTIC_CELL_CYCLE_PROCESS","GOBP_CHROMOSOME_SEGREGATION","GOBP_CELL_CYCLE","GOBP_CELLULAR_COMPONENT_DISASSEMBLY"
  ),
  MP7 = c(
      "GOBP_CELLULAR_RESPONSE_TO_STRESS","GOBP_DNA_METABOLIC_PROCESS","GOBP_DNA_REPAIR","GOBP_CELLULAR_RESPONSE_TO_DNA_DAMAGE_STIMULUS"
  )
)

In [ ]:
custom_palette <- colorRampPalette(c("#B9B4D5", "#8B83B9"))
custom_palette(3)

In [ ]:
lapply(names(go_list),function(MP_select){

    print(MP_select)
    data_plot <- top_p_GOBP[[MP_select]] %>%
        filter(pathway %in% go_list[[MP_select]]) %>%
        arrange(desc(padj)) %>%
        mutate(pathway = factor(pathway,levels = pathway %>% unique()))
    min_value <- data_plot$padj %>% {-log10(.) * 10 } %>% min() %>% ceiling() %>% {./10}
    max_value <- data_plot$padj %>% {-log10(.) * 10 } %>% max() %>% floor() %>% {./10}
    options(repr.plot.height = 5,repr.plot.width = 8)
    p <- ggplot(data_plot, aes(x = -log10(padj), y = pathway, fill = -log10(padj))) +
      geom_bar(stat = 'identity',width = 0.5) +
      geom_text(
          aes(x = 0.01, y = pathway,label = str_replace(pathway, "^\\s+", "") %>% str_to_title()),
          size = 7,angle = 0,hjust = 0,vjust = -2,color = 'black') +

      scale_fill_gradient2(
            name = '-Log10(p.adjust)',
            low = custom_palette(3)[1],mid = custom_palette(3)[2],high = custom_palette(3)[3],
            midpoint = (min_value + max_value)/2,
          breaks = c(min_value,(min_value + max_value)/2,max_value),
          labels = c(min_value,(min_value + max_value)/2,max_value)
        ) +
      guides(fill = guide_colorbar(title.position = "left", title.theme = element_text(angle = 90,vjust = 0.5))) +
      scale_x_continuous(expand = c(0.01,0)) +
      scale_y_discrete(expand = c(0,0.4)) +
      labs(y = 'Pathway',x = 'RichFactor') +
      theme_classic(base_size = 20) +
      coord_cartesian(clip = "off") +
      theme(
        plot.margin = unit(c(60, 0, 0, 0), "pt"),
        plot.title = element_text(hjust = 0.5,size = 32),

        legend.title = element_text(size = 24,hjust = 0),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.4, "cm"),
        legend.key.width = unit(1.2, "cm"),

        axis.text.x = element_blank(),
        axis.text.y = element_blank(),
        axis.title.x = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
      )
    return(p)
})

## Figure 4b


In [ ]:
data_module <- lapply(
    names(geneNMF.metaprograms$metaprograms.genes.weights),
    function(Module_use) {
        df <- geneNMF.metaprograms$metaprograms.genes.weights[[Module_use]] %>%
            as.data.frame() %>%
            tibble::rownames_to_column("Genes") %>%
            rename_all(~ c("Genes", "Module_score")) %>%
            mutate(Module = Module_use)
        return(df)
    }
) %>%
    do.call(rbind, .) %>%
    group_by(Module) %>%
    arrange(desc(Module_score), .by_group = TRUE) %>%
    mutate(Module = factor(Module, levels = paste("MP", 1:16, sep = ""))) %>%
    group_by(Genes) %>%
    mutate(n = n()) %>%
    filter(n < 2 | Module_score == max(Module_score)) %>%
    group_by(Module) %>%
    arrange(desc(Module_score)) %>%
    slice(if (cur_group()$Module == "MP16") 1:20 else 1:10)

data_module %>% dim()
data_module$Module %>% table()
data_module$Genes %>% duplicated() %>% table()
data_module %>% head()


In [ ]:
data_module <- lapply(
    names(geneNMF.metaprograms$metaprograms.genes.weights),
    function(Module_use) {
        df <- geneNMF.metaprograms$metaprograms.genes.weights[[Module_use]] %>%
            as.data.frame() %>%
            tibble::rownames_to_column("Genes") %>%
            rename_all(~ c("Genes", "Module_score")) %>%
            mutate(Module = Module_use)
        return(df)
    }
) %>%
    do.call(rbind, .) %>%
    group_by(Module) %>%
    arrange(desc(Module_score), .by_group = TRUE) %>%
    mutate(Module = factor(Module, levels = paste("MP", 1:16, sep = ""))) %>%
    group_by(Genes) %>%
    mutate(n = n()) %>%
    filter(n < 2 | Module_score == max(Module_score)) %>%
    group_by(Module) %>%
    arrange(desc(Module_score)) %>%
    slice(if (cur_group()$Module == "MP16") 1:20 else 1:10)

data_module %>% dim()
data_module$Module %>% table()
data_module$Genes %>% duplicated() %>% table()
data_module %>% head()


In [ ]:
DefaultAssay(seu) <- 'RNA'
Idents(seu) <- seu$annotation
seu

In [ ]:
data_plot <- DotPlot(object = seu,features = data_module$Genes %>% unique(),assay = 'RNA',group.by = 'annotation')$data %>%

    rename(Genes = features.plot) %>%
    mutate(
        Genes = Genes %>% as.character()
    ) %>%
    left_join(data_module %>% mutate(Genes = Genes %>% as.character()), by = 'Genes') %>%
    mutate(
        Genes = factor(Genes,levels = unique(data_module$Genes)),
        Module
    )
data_plot %>% dim()
data_plot %>% head()

In [ ]:
data_plot$avg.exp.scaled %>% max()
data_plot$avg.exp.scaled %>% min()

In [ ]:
options(repr.plot.width = 7,repr.plot.height = 18)
ggplot(
    data = data_plot %>%
        filter(Module %in% c('MP1','MP7','MP12','MP16')) %>%
    mutate(Module = factor(Module,levels = c('MP12','MP16','MP1','MP7'))),
    mapping = aes(x = id,y = Genes,color = avg.exp.scaled,size = pct.exp)
) +
    geom_point() +
    guides(
        color = guide_colorbar(title = 'Avg Expression',title.vjust = 1,title.position = "left",
      title.theme = element_text(angle = 90, vjust = 0.5))
      ) +
    scale_color_gradientn(
      colors = c('#2B86A6', '#FDF9D9', '#E57168', '#81141a'),

      limits = c(-1.5, 1.5),
      oob = scales::squish,
      breaks = c(-1.5, 0,1.5),
      labels = c(-1.5, 0,1.5)
    ) +
    facet_wrap( ~ Module,scales = 'free_y',ncol = 1,strip.position = 'left') +
    theme_classic() +
    theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(0.8, "cm"),
        axis.text.x = element_text(size = 24,angle = 45,hjust = 1,vjust = 1),
        axis.title = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
    )

In [ ]:
diffgene_all <- FindAllMarkers(object = seu,assay = 'RNA',slot = 'data',min.pct = 0,only.pos = TRUE)
diffgene_all %>% head()

In [ ]:
data_diff <- diffgene_all %>%
    group_by(cluster) %>%
    filter(p_val<0.01) %>%
    arrange(avg_log2FC,.by_group = TRUE) %>%

    rename('Genes' = 'gene','Module' = 'cluster') %>%
    mutate(
        Module = factor(Module,levels = paste('Subset ',1:4,sep = ''))
    ) %>%

    arrange(Module,avg_log2FC)
data_diff %>% dim()
data_diff %>% head()

In [ ]:
data_diff <- data_diff %>%
    filter(avg_log2FC>0.1)

In [ ]:
my36colors <-c(
  "#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b",
  "#e377c2","#7f7f7f","#bcbd22","#17becf","#aec7e8","#ffbb78",
  "#98df8a","#ff9896","#c5b0d5","#c49c94","#f7b6d2","#c7c7c7",
  "#dbdb8d","#9edae5","#7698b3","#d6616b","#a55194","#ce6dbd",
  "#756bb1","#8c6d31","#b5cf6b","#7b4173","#cedb9c","#6b6ecf",
  "#9c9ede","#bd9e39","#d9d9d9","#ad494a","#8ca252","#e7ba52"
) %>% sample(36,replace = FALSE)

In [ ]:
data_module_use <- data_module %>%
    filter(Module %in% c('MP1','MP7','MP12','MP16')) %>%
    mutate(Module = factor(Module,levels = c('MP12','MP16','MP1','MP7')))
data_module_use %>% dim()
data_module_use %>% head()

In [ ]:
data_diff_use <- data_diff %>%
  group_by(Genes) %>%
  mutate(n = n()) %>%
  filter(n < 2 | avg_log2FC == max(avg_log2FC)) %>%
  select(-n) %>%
  ungroup() %>%
  arrange(Module,desc(avg_log2FC))
data_diff %>%  dim()
data_diff_use %>%  dim()
data_diff_use %>%  head()

In [ ]:
data_plot <- DotPlot(object = seu,features = data_diff_use$Genes %>% unique(),assay = 'RNA',group.by = 'annotation')$data %>%

    rename(Genes = features.plot) %>%
    mutate(
        Genes = Genes %>% as.character()
    ) %>%
    left_join(data_module %>% mutate(Genes = Genes %>% as.character()), by = 'Genes') %>%
    mutate(
        Genes = factor(Genes,levels = data_diff_use$Genes %>% unique())
    )
data_plot %>% dim()
data_plot %>% head()

In [ ]:
data_plot <- data_plot %>%
    select(c('avg.exp.scaled','Genes','id')) %>%
    pivot_wider(
        names_from = 'id',
        values_from = 'avg.exp.scaled',
        values_fn = ~ mean(.x, na.rm = TRUE)
    ) %>%
    mutate(Genes = factor(Genes,levels = data_diff_use$Genes)) %>%
    arrange(Genes) %>%

    column_to_rownames('Genes') %>%
    select(paste('Subset',1:4))
data_plot %>% dim()
data_plot %>% head()

In [ ]:
# Figure 4B dependency.
group_df <- data_diff_use %>%
    mutate(group = Module) %>%
    arrange(group) %>%
    distinct(Genes, .keep_all = TRUE) %>%
    select(Genes, group)
rownames(group_df) <- group_df$Genes
group_vec <- factor(group_df$group, levels = paste("Subset", 1:4))


In [ ]:
# Figure 4B annotation.
modules <- unique(data_module_use$Module)
module_colors <- setNames(
    c('#DE9980',"#799D8E","#859ECA","#9687AD"),
    modules
)
genes_for_mark <- data_module_use$Genes[!grepl('ENSG', data_module_use$Genes)] %>% as.character()
gene_module_map <- data_module_use %>% 
    filter(Genes %in% genes_for_mark) %>%
    select(Genes, Module) %>% 
    mutate(Genes = factor(Genes,levels = rownames(data_plot)[which(rownames(data_plot) %in% genes_for_mark)])) %>% 
    arrange(Genes)
labels_colors <- module_colors[gene_module_map$Module]

row_anno <- rowAnnotation(
  link = anno_mark(
    at = which(rownames(data_plot) %in% genes_for_mark), 
    labels = rownames(data_plot)[which(rownames(data_plot) %in% genes_for_mark)],
    labels_gp = gpar(fontsize = 20, col = labels_colors)
  )
)

In [ ]:
# Figure 4B annotation.
module_colors <- setNames(
  colorRampPalette(RColorBrewer::brewer.pal(length(modules), "Set2"))(length(group_vec %>% levels())),
  group_vec %>% levels()
)
group_annotation <- rowAnnotation(
  group = anno_block(
    gp = gpar(fill = module_colors[levels(group_vec)]),
    labels = levels(group_vec),
    labels_gp = gpar(fontsize = 28,col = 'white'),
    labels_rot = 90
  ),
  show_annotation_name = FALSE,
  width = unit(10, "mm")
)
group_legend <- Legend(
  labels = levels(group_vec),
  legend_gp = gpar(fill = module_colors[levels(group_vec)]),
  title = "Group",
  title_gp = gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
  labels_gp = gpar(fontsize = 24, col = "black"),
  title_position = "leftcenter-rot",
  grid_width = unit(11, "mm"),
  grid_height = unit(12, "mm")
)

In [ ]:
data_plot_scaled <- t(scale(t(as.matrix(data_plot %>% dplyr::select(where(is.numeric))))))

In [ ]:
group_vec %>% unique()

In [ ]:
# Figure 4B annotation.
options(repr.plot.width = 12,repr.plot.height = 32)
ht <- Heatmap(
    matrix = data_plot %>% as.matrix(),
    col = colorRamp2(c(data_plot %>% min() %>% {./2},0, data_plot %>% max() %>% {.*1.2}),c("white", "skyblue", "darkblue")),
    left_annotation = group_annotation,
    row_split = group_vec,
    row_title_gp = gpar(fontsize = 0),
    column_title_gp = gpar(fontsize = 0),
    row_gap = unit(3, "mm"),
    column_gap = unit(2, "mm"),
    row_names_gp = gpar(fontsize = 26),
    column_names_gp = gpar(fontsize = 26),
    cluster_rows = TRUE,
    cluster_columns = FALSE,
    show_column_names = TRUE,
    show_row_names = TRUE,
    heatmap_legend_param = list(
        title = 'Relative Expression',
        title_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
        title_position = 'leftcenter-rot',
        legend_direction = 'vertical',
        legend_height = unit(80, units = "mm"),
        labels_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5)
    )
)
draw(ht + row_anno)

## Figure 4c


In [ ]:
DefaultAssay(seu) <- 'Protein'
seu <- NormalizeData(object = seu,normalization.method = "CLR", margin =2)
seu

In [ ]:
data_plot <- FetchData(object = seu,vars = c("ALOD4-pAbO","c-MYc-pAbO",'annotation')) %>%
    tidyr::pivot_longer(cols = c("ALOD4-pAbO","c-MYc-pAbO"),names_to = 'Protein',values_to = 'value') %>%
    rename(batch = annotation) %>%
    mutate(
        batch = factor(batch,levels = c('Subset 4','Subset 3','Subset 2','Subset 1') %>% rev()),
        Protein = Protein %>% str_remove('-pAbO'),
        Protein = case_when(
            Protein == 'c-MYc' ~ "MYC",
            TRUE ~ Protein
        )
    )
data_plot$Protein %>% unique()
data_plot %>% dim()
data_plot %>% head()

In [ ]:
data_plot$batch %>% unique()

In [ ]:
library(ggpubr)
library(paletteer)
options(repr.plot.width = 8.8,repr.plot.height = 7)
ggboxplot(
      data_plot,
      x="batch", y="value",
      color ="batch",
      width = 0.6,
      palette = c('#DE9980',"#799D8E","#859ECA","#9687AD"),
      add = "jitter",alpha = 0.9,
      xlab = F,  bxp.errorbar=T,
      bxp.errorbar.width=0.5,
      add.params = list(alpha = 0.2, size = 0.4),
      size=0.5, outlier.shape=NA,legend = "right") +
      guides(color = guide_legend(title = 'Group'))+
      stat_compare_means(
        label = "p.format",size = 6,hide.ns = TRUE,
        comparisons = (data_plot$batch %>% unique() %>% as.character()  %>% sort() %>% combn(m = 2,simplify = FALSE)),
        method = "wilcox.test",
      ) +
      facet_wrap(~Protein,ncol = 9,strip.position = 'left',scales = 'free') +

      theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(1.2, "cm"),
        axis.text.x = element_blank(),
        axis.title.y = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
      )

In [ ]:
library(ggpubr)
library(paletteer)
options(repr.plot.width = 8.8,repr.plot.height = 7)
ggboxplot(
      data_plot,
      x="batch", y="value",
      color ="batch",
      width = 0.6,
      palette = c('#DE9980',"#799D8E","#859ECA","#9687AD"),
      add = "jitter",alpha = 0.9,
      xlab = F,  bxp.errorbar=T,
      bxp.errorbar.width=0.5,
      add.params = list(alpha = 0.2, size = 0.4),
      size=0.5, outlier.shape=NA,legend = "right") +
      guides(color = guide_legend(title = 'Group'))+

      facet_wrap(~Protein,ncol = 9,strip.position = 'left',scales = 'free') +

      theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(1.2, "cm"),
        axis.text.x = element_blank(),
        axis.title.y = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
      )

## Figure 4d


In [ ]:
target_gene <- seu@assays$sgRNA %>% rownames() %>% str_remove('-sg.') %>% unique() %>% .[!(. %in% c('AAVS','NT1','NT2','topNA-number','sumNA-number','ratio-top-sum'))]
target_gene %>% length()
target_gene
DefaultAssay(seu) <- 'RNA'
seu

In [ ]:
colnames(seu@meta.data )

In [ ]:
target_gene %in% rownames(seu)

In [ ]:
data_plot <-  FetchData(seu, vars = c('annotation','MP1','MP7','MP12','MP16',target_gene),layer = 'data') %>%
    tibble::rownames_to_column('Sample') %>%
    tidyr::pivot_longer(cols = target_gene,names_to = 'Protein',values_to = 'Expression') %>%
    tidyr::pivot_longer(cols = c('MP1','MP7','MP12','MP16'),names_to = 'Module',values_to = 'UCell_module') %>%
    dplyr::filter(
        (annotation == 'Subset 1' & Module == 'MP12') |
        (annotation == 'Subset 2' & Module == 'MP16') |
        (annotation == 'Subset 3' & Module == 'MP1') |
        (annotation == 'Subset 4' & Module == 'MP7')
    )
data_plot %>% head()

In [ ]:
library(ggplot2)
library(ggpmisc)

In [ ]:
r2_table <- data_plot %>%
  group_by(Protein, annotation) %>%
  summarise(
    r2 = summary(lm(Expression ~ UCell_module))$r.squared,
    pval = summary(lm(Expression ~ UCell_module))$coefficients[2, 4],
    n = n()
  ) %>%
    arrange(desc(r2))
r2_table

In [ ]:
options(repr.plot.width = 42,repr.plot.height = 5*30)
ggplot(data = data_plot,aes(x = UCell_module,y = Expression)) +
    geom_point() +
    geom_smooth(method = "lm",formula = y ~ x, se = FALSE, color = "blue") +
    stat_poly_eq(
        aes(label = paste(after_stat(eq.label), ..rr.label.., ..p.value.label.., sep = "~~~")),
        formula = y ~ x,
        parse = TRUE,size = 8
    ) +
    facet_wrap(Protein ~ annotation,ncol = 4,strip.position = 'left',scales = 'free') +
    theme_classic() +
    theme(
        aspect.ratio = 1,
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(1.2, "cm"),
        axis.text.x = element_blank(),
        axis.title.y = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
    )

In [ ]:
data_plot_sub <- data_plot %>%
    filter(annotation == 'Subset 4' & Protein == 'ALOD4-pAbO')
data_plot_sub %>% head()

In [ ]:
data_plot_heatmap <- data_plot %>%
    group_by(annotation,Protein)%>%
    summarise(
        spearman_r = cor(Expression, UCell_module, method = "spearman", use = "complete.obs"),

        .groups = "drop"
    ) %>%
    arrange(annotation,desc(abs(spearman_r))) %>%
    group_by(annotation) %>%
    pivot_wider(names_from = Protein, values_from = spearman_r) %>%
    tibble::column_to_rownames('annotation') %>%
    as.matrix()
data_plot_heatmap %>% head()

In [ ]:
data_plot %>% head()

In [ ]:
library(ComplexHeatmap)
library(circlize)
library(dendextend)

In [ ]:
col_fun <- colorRamp2(
  c(0, data_plot_heatmap %>% max() %>% {./2}, data_plot_heatmap %>% max() %>% {.*1.2}),
  c("white", "skyblue", "darkblue"))

In [ ]:
options(repr.plot.width = 24,repr.plot.height = 4)

ht <- Heatmap(
  data_plot_heatmap,
  name = "Jaccard",
  col = col_fun,
  cluster_rows = TRUE,
  cluster_columns = TRUE,
  show_row_dend = TRUE,
  show_column_dend = TRUE,
  row_names_gp = gpar(fontsize = 6),
  column_names_gp = gpar(fontsize = 16)
)

ht_built <- draw(ht, merge_legend = TRUE)

row_dend <- row_dend(ht_built)
col_dend <- column_dend(ht_built)

row_hc <- as.hclust(row_dend)
col_hc <- as.hclust(col_dend)

k_row <- 4
k_col <- 8

row_dend <- as.dendrogram(row_hc)
col_dend <- as.dendrogram(col_hc)

row_dend_colored <- color_branches(row_dend, k = k_row)
col_dend_colored <- color_branches(col_dend, k = k_col)

row_groups <- cutree(row_hc, k = k_row)
col_groups <- cutree(col_hc, k = k_col)

In [ ]:
col_fun <- colorRamp2(
  c(data_plot_heatmap %>% min() %>% {.*1},0, data_plot_heatmap %>% max() %>% {.*1}),
  c("#26456E","white",  "#9C0824"))

In [ ]:
data_plot_heatmap

In [ ]:
options(repr.plot.width = 24,repr.plot.height = 7)

ht_all <- Heatmap(
    data_plot_heatmap,
    name = "Spearman",
    col = col_fun,na_col = "white",
    cell_fun = function(j,i,x,y,width,height,fill){
    grid.text(
        label = sprintf("%.2f",data_plot_heatmap[i,j]),
        x=x,y = y,gp = gpar(fontsize = 12,col = "black")
    )
    },
    cluster_rows = FALSE,
    cluster_columns = col_dend_colored,

    column_split = k_col,

    show_row_dend = TRUE,
    show_column_dend = TRUE,
    row_names_gp = gpar(fontsize = 26),
    column_names_gp = gpar(fontsize = 26),
    show_heatmap_legend = TRUE,
    heatmap_legend_param = list(
        title = 'Spearman',
        title_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
        title_position = 'leftcenter-rot',
        legend_direction = 'vertical',

        labels_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5)
    )
)

draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))

In [ ]:
DefaultAssay(seu) <- 'Protein'
seu <- NormalizeData(object = seu,normalization.method = "CLR", margin =2)
seu

In [ ]:
colnames(seu@meta.data )

In [ ]:
data_plot <-  FetchData(seu, vars = c('annotation','MP1','MP7','MP12','MP16',rownames(seu)),layer = 'data') %>%
    tibble::rownames_to_column('Sample') %>%
    tidyr::pivot_longer(cols = rownames(seu),names_to = 'Protein',values_to = 'Expression') %>%
    tidyr::pivot_longer(cols = c('MP1','MP7','MP12','MP16'),names_to = 'Module',values_to = 'UCell_module') %>%
    dplyr::filter(
        (annotation == 'Subset 1' & Module == 'MP12') |
        (annotation == 'Subset 2' & Module == 'MP16') |
        (annotation == 'Subset 3' & Module == 'MP1') |
        (annotation == 'Subset 4' & Module == 'MP7')
    )
data_plot %>% head()

In [ ]:
library(ggplot2)
library(ggpmisc)

In [ ]:
r2_table <- data_plot %>%
  group_by(Protein, annotation) %>%
  summarise(
    r2 = summary(lm(Expression ~ UCell_module))$r.squared,
    pval = summary(lm(Expression ~ UCell_module))$coefficients[2, 4],
    n = n()
  ) %>%
    arrange(desc(r2))
r2_table

In [ ]:
options(repr.plot.width = 42,repr.plot.height = 7*30)
ggplot(data = data_plot,aes(x = UCell_module,y = Expression)) +
    geom_point() +
    geom_smooth(method = "lm",formula = y ~ x, se = FALSE, color = "blue") +
    stat_poly_eq(
        aes(label = paste(after_stat(eq.label), ..rr.label.., ..p.value.label.., sep = "~~~")),
        formula = y ~ x,
        parse = TRUE,size = 8
    ) +
    facet_wrap(Protein ~ annotation,ncol = 4,strip.position = 'left',scales = 'free') +
    theme_classic() +
    theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(1.2, "cm"),
        axis.text.x = element_blank(),
        axis.title.y = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
    )

In [ ]:
data_plot_sub <- data_plot %>%
    filter(annotation == 'Subset 4' & Protein == 'ALOD4-pAbO')
data_plot_sub %>% head()

In [ ]:
data_plot_heatmap <- data_plot %>%
    group_by(annotation,Protein)%>%
    summarise(
        spearman_r = cor(Expression, UCell_module, method = "spearman", use = "complete.obs"),

        .groups = "drop"
    ) %>%
    arrange(annotation,desc(abs(spearman_r))) %>%
    group_by(annotation) %>%
    pivot_wider(names_from = Protein, values_from = spearman_r) %>%
    tibble::column_to_rownames('annotation') %>%
    as.matrix()
data_plot_heatmap %>% head()

In [ ]:
data_plot_pval <- data_plot %>%
  group_by(annotation, Protein) %>%
  summarise(
    pval = cor.test(Expression, UCell_module,
                    method = "spearman",
                    use = "complete.obs")$p.value,
    .groups = "drop"
  ) %>%
  arrange(annotation, desc(abs(pval))) %>%
  group_by(annotation) %>%
  pivot_wider(names_from = Protein, values_from = pval) %>%
  tibble::column_to_rownames("annotation") %>%
  as.matrix()

head(data_plot_pval)

In [ ]:
library(ComplexHeatmap)
library(circlize)
library(dendextend)

In [ ]:
col_fun <- colorRamp2(
  c(0, data_plot_heatmap %>% max() %>% {./2}, data_plot_heatmap %>% max() %>% {.*1.2}),
  c("white", "skyblue", "darkblue"))

In [ ]:
options(repr.plot.width = 24,repr.plot.height = 4)

ht <- Heatmap(
  data_plot_heatmap,
  name = "Jaccard",
  col = col_fun,
  cluster_rows = TRUE,
  cluster_columns = TRUE,
  show_row_dend = TRUE,
  show_column_dend = TRUE,
  row_names_gp = gpar(fontsize = 6),
  column_names_gp = gpar(fontsize = 16)
)

ht_built <- draw(ht, merge_legend = TRUE)

row_dend <- row_dend(ht_built)
col_dend <- column_dend(ht_built)

row_hc <- as.hclust(row_dend)
col_hc <- as.hclust(col_dend)

k_row <- 4
k_col <- 8

row_dend <- as.dendrogram(row_hc)
col_dend <- as.dendrogram(col_hc)

row_dend_colored <- color_branches(row_dend, k = k_row)
col_dend_colored <- color_branches(col_dend, k = k_col)

row_groups <- cutree(row_hc, k = k_row)
col_groups <- cutree(col_hc, k = k_col)

In [ ]:
col_fun <- colorRamp2(
  c(data_plot_heatmap %>% min() %>% {.*1.2},0, data_plot_heatmap %>% max() %>% {.*1.2}),
  c("#26456E","white",  "#9C0824"))

In [ ]:
data_plot_heatmap

In [ ]:
options(repr.plot.width = 24,repr.plot.height = 7)

ht_all <- Heatmap(
    data_plot_heatmap,
    name = "Spearman",
    col = col_fun,na_col = "white",
    cell_fun = function(j,i,x,y,width,height,fill){
    grid.text(
        label = sprintf("%.2f",data_plot_heatmap[i,j]),
        x=x,y = y,gp = gpar(fontsize = 12,col = "black")
    )
    },
    cluster_rows = FALSE,
    cluster_columns = col_dend_colored,

    column_split = k_col,

    show_row_dend = TRUE,
    show_column_dend = TRUE,
    row_names_gp = gpar(fontsize = 26),
    column_names_gp = gpar(fontsize = 26),
    show_heatmap_legend = TRUE,
    heatmap_legend_param = list(
        title = 'Spearman',
        title_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
        title_position = 'leftcenter-rot',
        legend_direction = 'vertical',

        labels_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5)
    )
)

draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))
draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))

In [ ]:
data_plot_heatmap %>% colnames()

In [ ]:
Protein_select <- c(
        'ALOD4-pAbO','pAKT-pAbO','pP65-pAbO','p-RPS6-pAbO','p-STAT3-pAbO','pERK-pAbO','pCREB-pAbO','pGSK3b-pAbO',
        'PDL1-pAbO','c-MYc-pAbO','SOX2-pAbO','Vimentin-pAbO','GPX4-pAbO'

     )

In [ ]:
subset_to_mp <- c(
    "Subset 1" = "MP12",
    "Subset 2" = "MP16",
    "Subset 3" = "MP1",
    "Subset 4" = "MP7"
)

display_order <- c("MP12", "MP16", "MP1", "MP7")

data_plot_pval_sub <- data_plot_pval %>%
    as.data.frame() %>%
    select(all_of(Protein_select)) %>%
    t() %>%
    as.data.frame()
colnames(data_plot_pval_sub) <- unname(subset_to_mp[colnames(data_plot_pval_sub)])
data_plot_pval_sub <- data_plot_pval_sub[, display_order, drop = FALSE]

data_plot_heatmap_sub <- data_plot_heatmap %>%
    as.data.frame() %>%
    t() %>%
    as.data.frame()
data_plot_heatmap_sub <- scale(
    data_plot_heatmap_sub,
    center = FALSE,
    scale = apply(abs(data_plot_heatmap_sub), 2, max)
) %>%
    t() %>%
    as.data.frame() %>%
    select(all_of(Protein_select)) %>%
    t()
colnames(data_plot_heatmap_sub) <- unname(subset_to_mp[colnames(data_plot_heatmap_sub)])
data_plot_heatmap_sub <- data_plot_heatmap_sub[, display_order, drop = FALSE]

data_plot_heatmap_sub


In [ ]:
data_plot_heatmap_sub <- data_plot_heatmap %>%
    as.data.frame() %>%

    t() %>% as.data.frame()
data_plot_heatmap_sub <- scale(data_plot_heatmap_sub, center = FALSE, scale = apply(abs(data_plot_heatmap_sub), 2, max)) %>%
    t() %>% as.data.frame() %>% select(all_of(Protein_select)) %>% t()

subset_to_mp <- c(
    "Subset 1" = "MP12",
    "Subset 2" = "MP16",
    "Subset 3" = "MP1",
    "Subset 4" = "MP7"
)
display_order <- c("MP12", "MP16", "MP1", "MP7")
colnames(data_plot_heatmap_sub) <- unname(subset_to_mp[colnames(data_plot_heatmap_sub)])
data_plot_heatmap_sub <- data_plot_heatmap_sub[, display_order, drop = FALSE]

data_plot_heatmap_sub



In [ ]:
col_fun <- colorRamp2(
  c(data_plot_heatmap_sub %>% min() %>% {.*1},0, data_plot_heatmap_sub %>% max() %>% {.*1}),
  c("#88A3D3","white",  "#BA6B6E"))

In [ ]:
options(repr.plot.width = 9,repr.plot.height = 14)

ht_all <- Heatmap(
    data_plot_heatmap_sub,
    name = "Spearman",
    col = col_fun,na_col = "white",
    cell_fun = function(j,i,x,y,width,height,fill){

        p <- data_plot_pval_sub[i, j]

        stars <- if (is.na(p)) {
          ""
        } else if (p < 0.001) {
          "(***)"
        } else if (p < 0.01) {
          "(**)"
        } else if (p < 0.05) {
          "(*)"
        } else {
          ""
        }
        lab <- sprintf("%.2f%s", data_plot_heatmap_sub[i, j], stars)
        grid.text(
            label = lab,
            x=x,y = y,gp = gpar(fontsize = 12,col = "black")
        )
    },
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    column_names_side = "top",
    row_names_side = "left",

    show_row_dend = TRUE,
    show_column_dend = TRUE,
    row_names_gp = gpar(fontsize = 26),
    column_names_gp = gpar(fontsize = 26),
    show_heatmap_legend = TRUE,
    heatmap_legend_param = list(
        title = 'Norm. correlation',
        title_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
        title_position = 'leftcenter-rot',
        legend_direction = 'vertical',
        legend_height = unit(70, units = "mm"),
        labels_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5)
    )
)

draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))
draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))

In [ ]:
options(repr.plot.width = 9,repr.plot.height = 14)

ht_all <- Heatmap(
    data_plot_heatmap_sub,
    name = "Spearman",
    col = col_fun,na_col = "white",
    cell_fun = function(j,i,x,y,width,height,fill){
        grid.text(
            label = sprintf("%.2f",data_plot_heatmap_sub[i,j]),
            x=x,y = y,gp = gpar(fontsize = 12,col = "black")
        )
    },
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    column_names_side = "top",
    row_names_side = "left",

    show_row_dend = TRUE,
    show_column_dend = TRUE,
    row_names_gp = gpar(fontsize = 26),
    column_names_gp = gpar(fontsize = 26),
    show_heatmap_legend = TRUE,
    heatmap_legend_param = list(
        title = 'Norm. correlation',
        title_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
        title_position = 'leftcenter-rot',
        legend_direction = 'vertical',
        legend_height = unit(70, units = "mm"),
        labels_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5)
    )
)

draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))
draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))

## Figure 4e


In [ ]:
# Figure 4e annotation.
metadata <- seu@meta.data %>%
    filter(batch == "Tatin")
MEs <- metadata %>%
    select(starts_with("MP"))

set.seed(123)
n_perm <- 10000

results_lm_perm <- lapply(
    colnames(MEs),
    function(module) {
        df <- data.frame(
            GeneScore = MEs[, module],
            perturbation = metadata$sgRNA_identity,
            batch = metadata$batch,
            nGene = metadata$nFeature_RNA
        ) %>%
            mutate(
                perturbation = case_when(
                    perturbation %in% c("NT1", "NT2", "AAVS") ~ "NT",
                    TRUE ~ perturbation
                ),
                perturbation = factor(perturbation),
                perturbation = relevel(perturbation, ref = "NT")
            ) %>%
            filter(!is.na(perturbation))

        model <- lm(
            GeneScore ~ perturbation + nGene,
            data = df
        )
        coefs <- summary(model)$coefficients
        perturbation_effects <- coefs[
            grep("perturbation", rownames(coefs)),
            , drop = FALSE
        ]
        pert_names <- rownames(perturbation_effects)
        orig_tvals <- abs(perturbation_effects[, "t value"])
        perm_tvals <- matrix(
            NA_real_,
            nrow = length(pert_names),
            ncol = n_perm,
            dimnames = list(pert_names, NULL)
        )

        for (i in seq_len(n_perm)) {
            df$perturbation_perm <- sample(df$perturbation)
            perm_model <- lm(
                GeneScore ~ perturbation_perm + nGene,
                data = df
            )
            perm_coefs <- summary(perm_model)$coefficients
            perm_perturbation <- perm_coefs[
                grep("perturbation_perm", rownames(perm_coefs)),
                , drop = FALSE
            ]
            rownames(perm_perturbation) <- sub(
                "_perm",
                "",
                rownames(perm_perturbation)
            )
            match_idx <- match(
                pert_names,
                rownames(perm_perturbation)
            )
            perm_tvals[, i] <- abs(
                perm_perturbation[match_idx, "t value"]
            )
        }

        empirical_p <- vapply(
            seq_along(pert_names),
            function(j) {
                mean(c(perm_tvals[j, ], orig_tvals[j]) >= orig_tvals[j])
            },
            numeric(1)
        )
        names(empirical_p) <- pert_names

        list(
            perturbation_effects = perturbation_effects,
            empirical_p = empirical_p
        )
    }
)
names(results_lm_perm) <- colnames(MEs)

lm_results_df <- do.call(
    rbind,
    lapply(names(results_lm_perm), function(module) {
        eff <- results_lm_perm[[module]]$perturbation_effects
        data.frame(
            rowname = paste(module, rownames(eff), sep = "_"),
            module = module,
            group = "tatin",
            term = rownames(eff),
            estimate = eff[, "Estimate"],
            t_value = eff[, "t value"],
            p_value = eff[, "Pr(>|t|)"],
            empirical_p = results_lm_perm[[module]]$empirical_p[
                rownames(eff)
            ]
        )
    })
)
lm_results_df$fdr_empirical_p <- p.adjust(
    lm_results_df$empirical_p,
    method = "BH"
)
write.csv(
    lm_results_df,
    file.path(SAFE_OUTPUT_ROOT, "Figure4e_regression_results.csv"),
    row.names = FALSE
)


In [ ]:
metadata  <- seu@meta.data %>%
    filter(batch == 'Tatin')
metadata$batch %>% table()
metadata %>% dim()
metadata %>% head()
metadata$batch %>% table()

In [ ]:
MEs <- metadata %>%
    select(starts_with("MP"))
MEs %>% head()

In [ ]:
if (RECOMPUTE_LONG_ANALYSES) {
  module <- colnames(MEs)[1]
  df <- data.frame(
      GeneScore = MEs[, module],
      perturbation = metadata$sgRNA_identity,
      batch = metadata$batch,
      nGene = metadata$nFeature_RNA
  ) %>%
  mutate(
      perturbation = case_when(
          perturbation %in% c('NT1','NT2','AAVS') ~ 'NT',
          TRUE ~ perturbation
      ),
      perturbation = factor(perturbation),
      perturbation = relevel(perturbation, ref = "NT")
  ) %>%
  filter(!is.na(perturbation))

  model <- lm(GeneScore ~ perturbation + nGene, data = df)
  coefs <- summary(model)$coefficients
  perturbation_effects <- coefs[grep("perturbation", rownames(coefs)), , drop = FALSE]
}

In [ ]:
if (RECOMPUTE_LONG_ANALYSES) {
  perturbation_effects %>% head()
}

In [ ]:
if (RECOMPUTE_LONG_ANALYSES) {
  set.seed(123)

  n_perm <- 10000

  results_lm_perm <- lapply(colnames(MEs), function(module){
    df <- data.frame(
      GeneScore = MEs[, module],
      perturbation = metadata$sgRNA_identity,
      batch = metadata$batch,
      nGene = metadata$nFeature_RNA
    ) %>%
      mutate(
          perturbation = case_when(
              perturbation %in% c('NT1','NT2','AAVS') ~ 'NT',
              TRUE ~ perturbation
          ),
          perturbation = factor(perturbation),
          perturbation = relevel(perturbation, ref = "NT")
      ) %>%
      filter(!is.na(perturbation))

    model <- lm(GeneScore ~ perturbation + nGene, data = df)
    coefs <- summary(model)$coefficients
    perturbation_effects <- coefs[grep("perturbation", rownames(coefs)), , drop = FALSE]

    pert_names <- rownames(perturbation_effects)

    orig_tvals <- abs(perturbation_effects[, "t value"])

    perm_tvals <- matrix(NA, nrow = length(pert_names), ncol = n_perm)
    rownames(perm_tvals) <- pert_names

    for (i in 1:n_perm) {

      df$perturbation_perm <- sample(df$perturbation)
      perm_model <- lm(GeneScore ~ perturbation_perm + nGene, data = df)
      perm_coefs <- summary(perm_model)$coefficients
      perm_perturbation <- perm_coefs[grep("perturbation_perm", rownames(perm_coefs)), , drop = FALSE]
      row.names(perm_perturbation) <- row.names(perm_perturbation) %>% str_remove('_perm')

      match_idx <- match(pert_names, rownames(perm_perturbation))
      perm_tvals[, i] <- abs(perm_perturbation[match_idx, "t value"])
    }

    empirical_p <- sapply(1:length(pert_names), function(j){
      mean(c(perm_tvals[j, ], orig_tvals[j]) >= orig_tvals[j])
    })
    names(empirical_p) <- pert_names

    list(
      perturbation_effects = perturbation_effects,
      empirical_p = empirical_p
    )
  })
}

In [ ]:
if (RECOMPUTE_LONG_ANALYSES) {
  names(results_lm_perm)
}

In [ ]:
if (RECOMPUTE_LONG_ANALYSES) {
  names(results_lm_perm) <- colnames(MEs)
  names(results_lm_perm)
}

In [ ]:
if (RECOMPUTE_LONG_ANALYSES) {
  results_lm_perm[[1]][2] %>% head()
}

In [ ]:
if (RECOMPUTE_LONG_ANALYSES) {
  lm_results_df <- do.call(rbind, lapply(names(results_lm_perm), function(mod){
    eff <- results_lm_perm[[mod]]$perturbation_effects
    pval_empirical <- results_lm_perm[[mod]]$empirical_p
    data.frame(
      module = mod,
      term = rownames(eff),
      estimate = eff[, "Estimate"],
      t_value = eff[, "t value"],
      p_value = eff[, "Pr(>|t|)"],
      empirical_p = pval_empirical[rownames(eff)]
    )
  }))

  lm_results_df$fdr_empirical_p <- p.adjust(lm_results_df$empirical_p, method = "BH")
  lm_results_df %>% head()
}

In [ ]:
# Figure 4e annotation.
lm_results_df %>% head()


In [ ]:
genes <- c("SREBF2", "HMGCR", "SQLE", "INSIG1",
           "LDLR", "NPC1L1", "NPC1",
           "APOB", "MTTP", "ABCA1", "ABCG1", "ABCG5", "ABCG8", "NR1H3",
           "SOAT1")
sgIdentity_sort <- lapply(genes,function(x) paste(x,c('-sg1','-sg2','-sg3'),sep = '')) %>% unlist()

In [ ]:
p.adjust(c(0.05,0.01,0.1,1), method = "BH")

In [ ]:
lm_results_df %>% colnames()

In [ ]:
(names(lm_results_df))

In [ ]:
library(stringr)
data_plot <- lm_results_df %>%
    mutate(
        p_log = -log10(empirical_p),

        is_sig = empirical_p < 0.01,
        term = term %>% str_remove('perturbation'),
        module = factor(module,levels = paste('MP',1:16,sep = ''))
    ) %>%
    group_by(module) %>%
    mutate(
        term = case_when(
            term %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ term
        ),
        term = factor(term,levels = sgIdentity_sort),

        estimate_norm = estimate
    ) %>%

    group_by(module) %>%
      mutate(estimate_scale =
               estimate / max(abs(estimate), na.rm = TRUE)
             ) %>%
      ungroup()  %>%
    filter(group == 'tatin') %>%
    column_to_rownames('rowname')
data_plot %>% head()

In [ ]:
colnames(data_plot)

In [ ]:
summary(data_plot$fdr_empirical_p)
summary(data_plot$estimate_norm)
summary(data_plot$estimate_scale)
data_plot %>%
    filter(term == 'NT') %>%
    pull(estimate_norm) %>% unique()

In [ ]:
data_plot <- data_plot %>%
    filter(term != 'NT')

In [ ]:
library(ggplot2)
options(repr.plot.width = 32, repr.plot.height = 10)
ggplot(data_plot, aes(x = term, y = module)) +

  geom_point(aes(size = p_log, color = estimate_scale)) +

  geom_point(data = subset(data_plot, is_sig),
             aes(size = p_log, color = estimate_scale),
             shape = 21, color = "black", stroke = 2,
             show.legend = FALSE) +
  scale_color_gradient2(low = "#1A5592",mid = "grey98", high = "#B83D3D") +

  scale_size_continuous(range = c(7, 10)) +
  labs(x = "Perturbation", y = "Cell Type", color = "Effect Size", fill = "Effect Size", size = "-log10(P-value)") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1),
    axis.title = element_blank(),
    axis.line = element_blank(),
    axis.ticks = element_blank(),
    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30, hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )

In [ ]:
data_plot_sub <- data_plot %>%
    filter(module %in% c('MP1','MP7','MP12','MP16')) %>%
    mutate(
        module = factor(module,levels = c('MP12','MP16','MP1','MP7') %>% rev()),
        facet_group = ifelse(term %in% sgIdentity_sort[1:21],yes = 'Group1',no = 'Group2')
    )
data_plot_sub %>% head()

In [ ]:
data_plot_sub <- data_plot_sub %>%
    group_by(module) %>%
      mutate(estimate_scale =
               estimate / max(abs(estimate), na.rm = TRUE)
             ) %>%
      ungroup()
data_plot_sub %>% head()

In [ ]:
data_plot_sub$estimate_scale %>% range()

In [ ]:
library(ggplot2)
options(repr.plot.width = 16, repr.plot.height = 8)
ggplot(data_plot_sub,aes(x = term, y = module)) +
  geom_point(aes(size = p_log, color = estimate_scale)) +
  geom_point(data = subset(data_plot_sub, is_sig),
             aes(size = p_log, color = estimate_scale),
             shape = 21, color = "black", stroke = 2,
             show.legend = FALSE) +

  scale_color_gradient2(
      low = "#1A5592",mid = "grey98", high = "#B83D3D",
      breaks = c(-0.8,0,1),
      labels = c(-0.8,0,1)
  ) +
  scale_size_continuous(range = c(7, 10)) +

  facet_wrap(~ facet_group,scales = 'free_x',ncol = 1) +
  labs(x = "Perturbation", y = "Cell Type", color = "Effect Size", fill = "Effect Size", size = "-log10(P-value)") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1),
    axis.title = element_blank(),
    axis.line = element_blank(),

    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30, hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )

## Figure 4f


In [ ]:
library(clusterProfiler)
library(msigdbr)
library(purrr)

In [ ]:
pathway_select <- c(
  "GOBP_STEROL_BIOSYNTHETIC_PROCESS",
  "GOBP_STEROL_METABOLIC_PROCESS",

  "GOBP_STEROID_METABOLIC_PROCESS",
  "GOBP_STEROID_BIOSYNTHETIC_PROCESS",

  "GOBP_STEROL_HOMEOSTASIS"

)

In [ ]:
gene_list <- msigdbr(species = 'Homo sapiens')
gene_list_sub <- gene_list %>%
    filter(gs_name %in% pathway_select) %>%
    dplyr::select(c('gs_name','gene_symbol')) %>%
    rename_all(~c('term','gene'))
gene_list_sub %>% head()

In [ ]:
gene_list_sub$gene %>% unique() %>% length()

In [ ]:
DefaultAssay(seurat_obj) <- 'RNA'
seurat_obj$sgRNA_type %>% unique()
seurat_obj

In [ ]:
adata_sub_use <- seurat_obj
adata_sub_use$sgRNA_type %>% table()
adata_sub_use

In [ ]:
gene_list_use <- gene_list_sub %>%
    filter(term %in% pathway_select) %>%
    unstack(gene ~ term)
gene_list_use %>% names() %>% length()
gene_list_use %>% names()
gene_list_use[[1]]

In [ ]:
adata_sub_use <- AddModuleScore(
    adata_sub_use,
    features = gene_list_use,
    ctrl = 100,seed = 1234,
    name = "pathway_")

In [ ]:
meta_data_use <- adata_sub_use@meta.data %>%
    rename_with(
        .fn = ~ names(gene_list_use),
        .cols = starts_with("pathway_")
    )
meta_data_use %>% head()
adata_sub_use@meta.data <- meta_data_use

In [ ]:
names(meta_data_use)

In [ ]:
for (column_name in c("function_type", "sgRNA_type_use")) {
    if (!column_name %in% colnames(meta_data_use)) {
        meta_data_use[[column_name]] <- NA_character_
    }
}

In [ ]:
pathway_matrix <- meta_data_use %>%
    dplyr::select(-c(
        'orig.ident','nCount_RNA','nFeature_RNA','nCount_Protein','nFeature_Protein','nCount_sgRNA',
        'nFeature_sgRNA','function_type','sgRNA_type_use','sgRNA_identity',
    ))  %>%
    group_by(batch,sgRNA_type) %>%
    summarise(across(everything(),median),.groups = 'drop') %>%
    mutate(group = paste(batch, sgRNA_type, sep = "_")) %>%
    column_to_rownames('group') %>%
    dplyr::select(-c('batch','sgRNA_type'))  %>% t()
pathway_matrix

In [ ]:
cor_mat <- cor(pathway_matrix, method = "spearman")

ranked_cor <- t(apply(cor_mat, 1, function(x) rank(x) / length(x)))

ranked_cor %>% dim()
ranked_cor

In [ ]:
row_id <- ranked_cor %>% rownames() %>% .[grepl('Tatin_',.)]
ranked_cor <- ranked_cor[row_id,row_id]
ranked_cor %>% head()

In [ ]:
ranked_cor

In [ ]:
ranked_cor %>% min()

In [ ]:
library(pheatmap)
options(repr.plot.width = 28,repr.plot.height = 22)
pheatmap(ranked_cor,fontsize = 18,cluster_rows = FALSE,cluster_cols = FALSE)

In [ ]:
library(igraph)

edges <- which(upper.tri(ranked_cor) & ranked_cor > 0.6, arr.ind = TRUE)
edge_list <- data.frame(
  from = rownames(ranked_cor)[edges[,1]],
  to = colnames(ranked_cor)[edges[,2]],
  weight = ranked_cor[edges]
)

In [ ]:
edges

In [ ]:
pathway_batches <- data.frame(name = rownames(ranked_cor)) %>%
    mutate(pathway = name) %>%
    separate(col = pathway,into = c('batch','sgRNA_identity'))
pathway_batches

In [ ]:
g <- graph_from_data_frame(edge_list, directed = FALSE)

V(g)$batch <- pathway_batches$batch[match(V(g)$name, pathway_batches$name)]
g %>% head()

In [ ]:
batch_levels <- unique(V(g)$batch)
batch_colors <- setNames(rainbow(length(batch_levels)), batch_levels)

vertex_colors <- batch_colors[V(g)$batch]
vertex_colors

In [ ]:
options(repr.plot.width = 22,repr.plot.height = 22)

plot(g,
     layout = layout_with_fr(g),
     vertex.label.cex = 3,
     vertex.size = 2,
     edge.width = E(g)$weight * 5,
     vertex.color = vertex_colors,
     vertex.label.color = "black",
     vertex.label.dist = 0.7,
     vertex.label.degree = -pi / 2,
     edge.color = "gray")